In [45]:
import numpy as np

# Исходные данные
A = np.array([
    [21, -6, -9, -4],
    [-6, 20, -4, 2],
    [-2, -7, -20, 3],
    [4, 9, 6, 24]
], dtype=float)
b = np.array([127, -144, 236, -5], dtype=float)
eps = 1e-6
n = len(b)



In [46]:
# Приведение к виду x = beta + alpha * x (формула 1.18)
alpha = np.zeros((n, n))
beta = np.zeros(n)
for i in range(n):
    beta[i] = b[i] / A[i, i]
    for j in range(n):
        if i != j:
            alpha[i, j] = -A[i, j] / A[i, i]

print('alpha:', alpha)
print('beta:', beta)



alpha: [[ 0.          0.28571429  0.42857143  0.19047619]
 [ 0.3         0.          0.2        -0.1       ]
 [-0.1        -0.35        0.          0.15      ]
 [-0.16666667 -0.375      -0.25        0.        ]]
beta: [  6.04761905  -7.2        -11.8         -0.20833333]


In [47]:
# Норма матрицы — максимальная сумма модулей элементов строки.
# Для вектора используется согласованная норма max |x_i|.
alpha_norm = np.max(np.sum(np.abs(alpha), axis=1))
alpha_norm


np.float64(0.9047619047619047)

In [48]:
# Верхняя треугольная часть alpha для оценки погрешности метода Зейделя
C = np.triu(alpha)
C_norm = np.max(np.sum(np.abs(C), axis=1))

C_norm

np.float64(0.9047619047619047)

In [52]:
# Сравнение числа итераций при разной точности
print("\nТочность, Простые итерации, Метод Зейделя")
for tolerance in [1e-2, 1e-4, eps]:
    # Метод простых итераций, начальное приближение x^(0) = beta
    x = beta.copy()
    error = np.inf
    iterations = 0
    while error > tolerance:
        x_prev = x.copy()
        x = beta + alpha @ x_prev
        iterations += 1
        difference = np.max(np.abs(x - x_prev))
        if alpha_norm < 1:
            error = alpha_norm / (1 - alpha_norm) * difference
        else:
            error = difference
    x_iterations = x.copy()
    error_iterations = error

    # Метод Зейделя: уже найденные компоненты берутся с текущей итерации
    x = beta.copy()
    error = np.inf
    seidel_iterations = 0
    while error > tolerance:
        x_prev = x.copy()
        for i in range(n):
            s = 0
            for j in range(i):
                s += alpha[i, j] * x[j]
            for j in range(i, n):
                s += alpha[i, j] * x_prev[j]
            x[i] = beta[i] + s
        seidel_iterations += 1
        difference = np.max(np.abs(x - x_prev))
        if alpha_norm < 1:
            error = C_norm / (1 - alpha_norm) * difference
        else:
            error = difference
    x_seidel = x.copy()
    print(tolerance, iterations, seidel_iterations)




Точность, Простые итерации, Метод Зейделя
0.01 12 7
0.0001 19 9
1e-06 25 12


In [53]:
print("\nРешение методом простых итераций:")
print(x_iterations)
print("Оценка погрешности:", error_iterations)




Решение методом простых итераций:
[ 0.99999996 -9.00000004 -7.99999999  5.00000004]
Оценка погрешности: 8.765652497189069e-07


In [51]:
print("\nРешение методом Зейделя:")
print(x_seidel)
print("Оценка погрешности:", error)



Решение методом Зейделя:
[ 1. -9. -8.  5.]
Оценка погрешности: 1.5061385960635202e-07
